In [5]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load datasets
df1 = pd.read_csv(r"/home/manem/Qwen-3VL-Testing/descriptions/after-finetuning/train/train_descriptions_realism_unsloth.csv")
df2 = pd.read_csv(r"/home/manem/Qwen-3VL-Testing/REALM-desc/train/image_descriptions_processed.csv")

# Rename for consistency
df2 = df2.rename(columns={"file_name": "image_path",
                          "description": "explanation"})

# Merge on filename
merged = pd.merge(df1, df2, on="image_path", suffixes=("_d1", "_d2"))

# -----------------------------
# Label Similarity
# -----------------------------

def normalize_label(label):
    label = str(label).strip().lower()
    if label == "yes":
        return 1.0
    elif label == "no":
        return 0.0
    elif label == "somewhat":
        return 0.5
    else:
        return 0.0

merged["label_score_d1"] = merged["unrealistic_d1"].apply(normalize_label)
merged["label_score_d2"] = merged["unrealistic_d2"].apply(normalize_label)

# Label similarity (1 - absolute difference)
merged["label_similarity"] = 1 - abs(
    merged["label_score_d1"] - merged["label_score_d2"]
)

# -----------------------------
# Text Similarity
# -----------------------------

model = SentenceTransformer("all-mpnet-base-v2")

emb1 = model.encode(merged["explanation_d1"].astype(str).tolist())
emb2 = model.encode(merged["explanation_d2"].astype(str).tolist())

text_similarities = []

for i in range(len(emb1)):
    sim = cosine_similarity(
        [emb1[i]],
        [emb2[i]]
    )[0][0]
    text_similarities.append(sim)

merged["text_similarity"] = text_similarities

# -----------------------------
# Final Document Similarity
# -----------------------------

merged["document_similarity"] = (
    merged["label_similarity"] + merged["text_similarity"]
) / 2

# Keep only required columns
result = merged[[
    "image_path",
    "label_similarity",
    "text_similarity",
    "document_similarity"
]]

print(result)
print(result.describe())


    image_path  label_similarity  text_similarity  document_similarity
0       f1.png               0.5         0.510764             0.505382
1      f10.png               0.5         0.362645             0.431323
2     f100.png               0.5         0.601319             0.550659
3     f101.png               1.0         0.867120             0.933560
4     f102.png               0.5         0.687989             0.593995
..         ...               ...              ...                  ...
505    r95.png               0.5         0.609552             0.554776
506    r96.png               1.0         0.935553             0.967776
507    r97.png               0.5         0.530755             0.515378
508    r98.png               0.0         0.705348             0.352674
509    r99.png               0.0         0.472780             0.236390

[510 rows x 4 columns]
       label_similarity  text_similarity  document_similarity
count        510.000000       510.000000           510.000000


In [6]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load datasets
df1 = pd.read_csv(r"/home/manem/Qwen-3VL-Testing/descriptions/before-finetuning/train/train_descriptions_realism_unsloth.csv")
df2 = pd.read_csv(r"/home/manem/Qwen-3VL-Testing/REALM-desc/train/image_descriptions_processed.csv")

# Rename for consistency
df2 = df2.rename(columns={"file_name": "image_path",
                          "description": "explanation"})

# Merge on filename
merged = pd.merge(df1, df2, on="image_path", suffixes=("_d1", "_d2"))

# -----------------------------
# Label Similarity
# -----------------------------

def normalize_label(label):
    label = str(label).strip().lower()
    if label == "yes":
        return 1.0
    elif label == "no":
        return 0.0
    elif label == "somewhat":
        return 0.5
    else:
        return 0.0

merged["label_score_d1"] = merged["unrealistic_d1"].apply(normalize_label)
merged["label_score_d2"] = merged["unrealistic_d2"].apply(normalize_label)

# Label similarity (1 - absolute difference)
merged["label_similarity"] = 1 - abs(
    merged["label_score_d1"] - merged["label_score_d2"]
)

# -----------------------------
# Text Similarity
# -----------------------------

model = SentenceTransformer("all-mpnet-base-v2")

emb1 = model.encode(merged["explanation_d1"].astype(str).tolist())
emb2 = model.encode(merged["explanation_d2"].astype(str).tolist())

text_similarities = []

for i in range(len(emb1)):
    sim = cosine_similarity(
        [emb1[i]],
        [emb2[i]]
    )[0][0]
    text_similarities.append(sim)

merged["text_similarity"] = text_similarities

# -----------------------------
# Final Document Similarity
# -----------------------------

merged["document_similarity"] = (
    merged["label_similarity"] + merged["text_similarity"]
) / 2

# Keep only required columns
result1 = merged[[
    "image_path",
    "label_similarity",
    "text_similarity",
    "document_similarity"
]]

print(result1)

print(result1.describe())

    image_path  label_similarity  text_similarity  document_similarity
0       f1.png               1.0         0.767242             0.883621
1      f10.png               1.0         0.470675             0.735337
2     f100.png               0.5         0.706560             0.603280
3     f101.png               0.5         0.766086             0.633043
4     f102.png               1.0         0.646800             0.823400
..         ...               ...              ...                  ...
505    r95.png               1.0         0.578308             0.789154
506    r96.png               1.0         0.677123             0.838561
507    r97.png               1.0         0.756344             0.878172
508    r98.png               0.0         0.783646             0.391823
509    r99.png               1.0         0.506099             0.753050

[510 rows x 4 columns]
       label_similarity  text_similarity  document_similarity
count        510.000000       510.000000           510.000000


In [5]:
import pandas as pd
import numpy as np
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')
nltk.download('punkt_tab') 
nltk.download('wordnet')
nltk.download('omw-1.4')

# Load datasets
df1 = pd.read_csv("/home/manem/Qwen-3VL-Testing/descriptions/after-finetuning/train/train_descriptions_realism_unsloth.csv")
df2 = pd.read_csv("/home/manem/Qwen-3VL-Testing/REALM-desc/train/image_descriptions_processed.csv")

df2 = df2.rename(columns={
    "file_name": "image_path",
    "description": "explanation"
})

merged = pd.merge(df1, df2, on="image_path", suffixes=("_d1", "_d2"))

# -----------------------------
# Label Similarity
# -----------------------------

def normalize_label(label):
    mapping = {
        "yes": 1.0,
        "somewhat": 0.5,
        "no": 0.0
    }
    return mapping.get(str(label).strip().lower(), np.nan)

merged["label_score_d1"] = merged["unrealistic_d1"].apply(normalize_label)
merged["label_score_d2"] = merged["unrealistic_d2"].apply(normalize_label)

merged = merged.dropna(subset=["label_score_d1", "label_score_d2"])

merged["label_similarity"] = 1 - abs(
    merged["label_score_d1"] - merged["label_score_d2"]
)

# -----------------------------
# METEOR Text Similarity
# -----------------------------

meteor_scores = []

for ref, hyp in zip(
    merged["explanation_d2"].astype(str),
    merged["explanation_d1"].astype(str)
):
    ref_tokens = word_tokenize(ref.lower())
    hyp_tokens = word_tokenize(hyp.lower())

    score = meteor_score([ref_tokens], hyp_tokens)
    meteor_scores.append(score)

merged["meteor_similarity"] = meteor_scores

# -----------------------------
# Final Document Similarity
# -----------------------------

ALPHA = 0.3
BETA = 0.7

merged["document_similarity"] = (
    ALPHA * merged["label_similarity"] +
    BETA * merged["meteor_similarity"]
)

result = merged[[
    "image_path",
    "label_similarity",
    "meteor_similarity",
    "document_similarity"
]]

print(result)
print(result.describe())


[nltk_data] Downloading package punkt to /home/manem/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/manem/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /home/manem/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/manem/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


    image_path  label_similarity  meteor_similarity  document_similarity
0       f1.png               0.5           0.269324             0.338527
1      f10.png               0.5           0.136054             0.245238
2     f100.png               0.5           0.092937             0.215056
3     f101.png               1.0           0.407393             0.585175
4     f102.png               0.5           0.234502             0.314151
..         ...               ...                ...                  ...
505    r95.png               0.5           0.250000             0.325000
506    r96.png               1.0           0.807102             0.864971
507    r97.png               0.5           0.148026             0.253618
508    r98.png               0.0           0.195313             0.136719
509    r99.png               0.0           0.155709             0.108997

[510 rows x 4 columns]
       label_similarity  meteor_similarity  document_similarity
count        510.000000         510.

In [2]:
import pandas as pd
import numpy as np
from bert_score import score

# Load datasets
df1 = pd.read_csv("/home/manem/Qwen-3VL-Testing/descriptions/after-finetuning/train/train_descriptions_realism_unsloth.csv")
df2 = pd.read_csv("/home/manem/Qwen-3VL-Testing/REALM-desc/train/image_descriptions_processed.csv")

df2 = df2.rename(columns={
    "file_name": "image_path",
    "description": "explanation"
})

merged = pd.merge(df1, df2, on="image_path", suffixes=("_d1", "_d2"))

# Drop NaNs just in case
merged = merged.dropna(subset=["explanation_d1", "explanation_d2"])

# Convert to list
candidates = merged["explanation_d1"].astype(str).tolist()
references = merged["explanation_d2"].astype(str).tolist()

# Compute BERTScore
P, R, F1 = score(
    candidates,
    references,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",  # strong model
    verbose=True
)

# Add scores to dataframe
merged["bert_precision"] = P.numpy()
merged["bert_recall"] = R.numpy()
merged["bert_f1"] = F1.numpy()

# Print overall average
print("Average BERTScore:")
print("Precision:", merged["bert_precision"].mean())
print("Recall:", merged["bert_recall"].mean())
print("F1:", merged["bert_f1"].mean())


/home/manem/miniconda3/envs/qwen3-vl-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


calculating scores...
computing bert embedding.


100%|██████████| 16/16 [00:05<00:00,  2.81it/s]


computing greedy matching.


100%|██████████| 8/8 [00:00<00:00, 15.10it/s]

done in 6.24 seconds, 81.78 sentences/sec
Average BERTScore:
Precision: 0.7082607
Recall: 0.70602137
F1: 0.70679283


In [3]:
import pandas as pd
import numpy as np
from bert_score import score

# Load datasets
df1 = pd.read_csv("/home/manem/Qwen-3VL-Testing/descriptions/before-finetuning/train/train_descriptions_realism_unsloth.csv")
df2 = pd.read_csv("/home/manem/Qwen-3VL-Testing/REALM-desc/train/image_descriptions_processed.csv")

df2 = df2.rename(columns={
    "file_name": "image_path",
    "description": "explanation"
})

merged = pd.merge(df1, df2, on="image_path", suffixes=("_d1", "_d2"))

# Drop NaNs just in case
merged = merged.dropna(subset=["explanation_d1", "explanation_d2"])

# Convert to list
candidates = merged["explanation_d1"].astype(str).tolist()
references = merged["explanation_d2"].astype(str).tolist()

# Compute BERTScore
P, R, F1 = score(
    candidates,
    references,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",  # strong model
    verbose=True
)

# Add scores to dataframe
merged["bert_precision"] = P.numpy()
merged["bert_recall"] = R.numpy()
merged["bert_f1"] = F1.numpy()

# Print overall average
print("Average BERTScore:")
print("Precision:", merged["bert_precision"].mean())
print("Recall:", merged["bert_recall"].mean())
print("F1:", merged["bert_f1"].mean())


calculating scores...
computing bert embedding.


100%|██████████| 16/16 [00:03<00:00,  4.40it/s]


computing greedy matching.


100%|██████████| 8/8 [00:00<00:00, 106.00it/s]

done in 3.72 seconds, 136.94 sentences/sec
Average BERTScore:
Precision: 0.6648427
Recall: 0.6670858
F1: 0.6657056


In [4]:
import pandas as pd
import numpy as np
from bert_score import score

# Load datasets
df1 = pd.read_csv("/home/manem/Qwen-3VL-Testing/descriptions/before-finetuning/test/test_descriptions_realism_unsloth.csv")
df2 = pd.read_csv("/home/manem/Qwen-3VL-Testing/REALM-desc/test/image_descriptions_processed.csv")

df2 = df2.rename(columns={
    "file_name": "image_path",
    "description": "explanation"
})

merged = pd.merge(df1, df2, on="image_path", suffixes=("_d1", "_d2"))

# Drop NaNs just in case
merged = merged.dropna(subset=["explanation_d1", "explanation_d2"])

# Convert to list
candidates = merged["explanation_d1"].astype(str).tolist()
references = merged["explanation_d2"].astype(str).tolist()

# Compute BERTScore
P, R, F1 = score(
    candidates,
    references,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",  # strong model
    verbose=True
)

# Add scores to dataframe
merged["bert_precision"] = P.numpy()
merged["bert_recall"] = R.numpy()
merged["bert_f1"] = F1.numpy()

# Print overall average
print("Average BERTScore:")
print("Precision:", merged["bert_precision"].mean())
print("Recall:", merged["bert_recall"].mean())
print("F1:", merged["bert_f1"].mean())


calculating scores...
computing bert embedding.


100%|██████████| 3/3 [00:00<00:00,  4.51it/s]


computing greedy matching.


100%|██████████| 2/2 [00:00<00:00, 130.81it/s]

done in 0.69 seconds, 130.57 sentences/sec
Average BERTScore:
Precision: 0.6625937
Recall: 0.6638376
F1: 0.6629158


In [5]:
import pandas as pd
import numpy as np
from bert_score import score

# Load datasets
df1 = pd.read_csv("/home/manem/Qwen-3VL-Testing/descriptions/after-finetuning/test/test_descriptions_realism_unsloth.csv")
df2 = pd.read_csv("/home/manem/Qwen-3VL-Testing/REALM-desc/test/image_descriptions_processed.csv")

df2 = df2.rename(columns={
    "file_name": "image_path",
    "description": "explanation"
})

merged = pd.merge(df1, df2, on="image_path", suffixes=("_d1", "_d2"))

# Drop NaNs just in case
merged = merged.dropna(subset=["explanation_d1", "explanation_d2"])

# Convert to list
candidates = merged["explanation_d1"].astype(str).tolist()
references = merged["explanation_d2"].astype(str).tolist()

# Compute BERTScore
P, R, F1 = score(
    candidates,
    references,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",  # strong model
    verbose=True
)

# Add scores to dataframe
merged["bert_precision"] = P.numpy()
merged["bert_recall"] = R.numpy()
merged["bert_f1"] = F1.numpy()

# Print overall average
print("Average BERTScore:")
print("Precision:", merged["bert_precision"].mean())
print("Recall:", merged["bert_recall"].mean())
print("F1:", merged["bert_f1"].mean())


calculating scores...
computing bert embedding.


100%|██████████| 3/3 [00:00<00:00,  4.08it/s]


computing greedy matching.


100%|██████████| 2/2 [00:00<00:00, 136.74it/s]


done in 0.75 seconds, 119.35 sentences/sec
Average BERTScore:
Precision: 0.7149475
Recall: 0.71187127
F1: 0.7129093
